# Comparing Six Reconstructor CNN Architectures

Every reconstructor architecture used anywhere in this repo (`SimpleNet`, `Rama`, `Papyrus1stStage`/`Papyrus2ndStage` in `AI4AO/PhaseEstimators.py`, `PupilCNN` in `DualSensorFusion.ipynb`, `PWFSNet` in `TwoStageAO.ipynb`/`05_TrainingAReconstructor.ipynb`) was picked once, for one notebook, and never benchmarked against an alternative under identical conditions. This notebook builds six small CNN reconstructors that only differ in how they handle a modulated Pyramid WFS's 4 pupil images, trains each following the exact same procedure as `Tutorials/basics/05_TrainingAReconstructor.ipynb` (`AI4AO.Trainer.Trainer`, same instrument, same loss, same `TrainRunNb`), and compares them on training-loss curves and on seeded closed-loop steady-state residual wavefront error (nm RMS). See `Ideas/07-cnn-architecture-comparison.md` for the full motivation and design discussion.

This is a single-frame reconstruction comparison -- no temporal window, no multi-sensor fusion (that's `Ideas/05-temporal-predictive-reconstructor.md`'s and `Ideas/04-concurrent-dual-sensor-fusion.md`'s territory respectively). Every architecture here sees exactly one WFS frame at the current tick.

**All six architectures are sized to roughly the same trainable-parameter budget (~0.9M)**, using `num_downsamples` below to pick each architecture's depth from the pupil crop's actual resolution (`Nout`, `2*Nout` for the retiled one) rather than a fixed, possibly-too-shallow pooling depth -- an equal-parameter, equal-depth-from-resolution comparison isolates the effect of *how* each architecture processes the 4 pupil channels, rather than confounding it with one network simply having more capacity or a shallower receptive field than another.

**The six architectures**, each isolating one design axis:

1. **`ClassicCNN`** -- a plain, non-grouped `Conv2d` stack (VGG-style stages) directly on the 4 registered pupil channels (styled on `Rama`'s existing encoder). No per-pupil structure, no positional information -- the "channels are just channels" baseline.
2. **`StemEncoderCNN`** -- the `PupilCNN`/`PWFSNet` idea, deepened: a `groups=4` stem processes each pupil image independently before a shared encoder mixes information across pupils.
3. **`CoordConvCNN`** -- `ClassicCNN`'s exact conv-stage structure, with two extra channels (normalized x/y pixel-coordinate grids, CoordConv-style) concatenated to the 4 pupil channels before the first stage.
4. **`DetiledCNN`** -- the same 4 pupil crops `FramePreprocess` already produces, retiled into one `(2*Nout, 2*Nout)` single-channel image in their true physical quadrant layout, fed to a `SimpleNet`-style single-channel conv stack -- one stage deeper than the others, since the retiled image has twice the linear resolution.
5. **`RecommendedCNN`** -- `StemEncoderCNN`'s grouped-conv stem + `CoordConvCNN`'s coordinate channels (injected only where the stem's output first merges across pupils) + a lightweight squeeze-and-excitation block before the head, letting the network adaptively reweight the merged per-pupil features.
6. **`ResNetCNN`** -- `CoordConvCNN`'s input convention (plain convs + coordinate channels), reorganized into ResNet-style residual stages: a strided conv downsamples between stages, and each stage refines its features through a couple of residual blocks before the next downsample.


In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda'  # set to "cpu" if CUDA is not available


## Configuration and instrument

`CNNArchitectureComparison_params.py` holds one `WFSParams`/`AtmosParams`/`LoopParams`/`DMParams`/`TrainParams` for a single fictional-demo, nominal-geometry modulated Pyramid WFS + DM -- no real bench behind it, following the same "no real bench" pattern `TwoStageAO.ipynb`/`DualSensorFusion.ipynb` already establish for a cross-cutting technique. `wfs` and `dm` are built fresh (not loaded from a saved calibration) and frozen (`.eval()`), since only each architecture's reconstructor weights are ever trained here. `DMParams["Nmodes"]` is a fixed integer read directly by `dm.MakeZernikeM2C()` and by every architecture's linear head, matching `05_TrainingAReconstructor.ipynb`'s own params-file convention.

In [ ]:
paramfile = 'CNNArchitectureComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = "../../Data/CNNArchitectureComparison/"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()
dm.requires_grad_(False)

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")


## Retiling the four pupil images (for `DetiledCNN`)

`FramePreprocess.GetTrainingPupils` already crops and registers each pupil image independently into its own channel, `(B, 4, Nout, Nout)`. `retile_pupils` reassembles those same 4 channels into one `(B, 1, 2*Nout, 2*Nout)` single-channel image, in the Pyramid's true physical quadrant layout -- no new crop/registration code, purely a reassembly of data `FramePreprocess` already produces.

**Quadrant order**, derived from `PyramidWFS.GetPupilCenter`: `sign_tensor = -[[1,1],[-1,1],[-1,-1],[1,-1]]`, and `FramePreprocess.py`'s own `centers` convention is `(y, x)` -- so pupil index 0 sits at sign `(-1,-1)` (**top-left**), index 1 at `(1,-1)` (**bottom-left**), index 2 at `(1,1)` (**bottom-right**), index 3 at `(-1,1)` (**top-right**). This derivation is confirmed from the code, but a wrong tiling order here would not raise an exception -- it would just silently train on a physically-scrambled-but-plausible-looking single-channel image. The cell below validates it by eye: the retiled image's four quadrants should visibly match the raw WFS frame's own four-pupil layout.

In [ ]:
def retile_pupils(frames):
    """Reassemble FramePreprocess's 4 registered pupil channels (B,4,Nout,Nout)
    into one (B,1,2*Nout,2*Nout) single-channel image, in PyramidWFS's true
    quadrant layout (see PyramidWFS.GetPupilCenter): pupil 0 top-left,
    pupil 1 bottom-left, pupil 2 bottom-right, pupil 3 top-right."""
    top = torch.cat([frames[:, 0:1], frames[:, 3:4]], dim=-1)
    bottom = torch.cat([frames[:, 1:2], frames[:, 2:3]], dim=-1)
    return torch.cat([top, bottom], dim=-2)


with torch.no_grad():
    batch = dataset[0]
    wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])
    check_frames = wfs(batch["opd"], batch["pupil"])
    check_preprocessed = framePreprocessor.ProcessFrame(check_frames, add_pupil_noise=False)
    check_retiled = retile_pupils(check_preprocessed)

imshow_multiple([
    {"tensor": check_frames[0], "title": "Raw WFS frame"},
    {"tensor": check_retiled[0], "title": "Retiled single-channel image"},
])


## The six architectures

Each maps `(B, 4, Nout, Nout) -> (B, Nmodes)` (architecture 4 internally retiles to a single channel first), so any of them can be dropped straight into `Trainer` unchanged.

`num_downsamples` and `conv_stage` below are the shared building blocks every plain-conv architecture uses to pick its own depth from the actual pupil-crop resolution: at the default `Nout=46`, a feature map can take 4 stride-2 poolings before dropping below 2 pixels (`46 -> 23 -> 11 -> 5 -> 2`), so every architecture below uses 4 downsampling stages (`DetiledCNN`'s retiled `2*Nout=92` input gets a 5th, since it starts at twice the resolution) -- deeper than a fixed 2-pooling design would allow, and automatically adapted if `Nout` ever changes (e.g. a different `Extract_pupils_pad`/`Bin_factor`).

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can
    take before dropping below min_size pixels -- lets every architecture
    below downsample as deep as the actual pupil-crop resolution allows,
    rather than a fixed, possibly-too-shallow number of pooling stages."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, activation=nn.LeakyReLU):
    """Two 3x3 convs at out_channels, then a 2x downsampling MaxPool --
    one VGG-style stage, reused by every plain-conv architecture below."""
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.MaxPool2d(2),
    ]


In [ ]:
class ClassicCNN(nn.Module):
    """Plain, non-grouped Conv2d stack -- channels are just channels, no
    per-pupil structure and no positional information. Styled on Rama's
    existing encoder in AI4AO/PhaseEstimators.py, but deeper: num_downsamples
    picks the number of VGG-style stages from Nout itself."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=28):
        super().__init__()

        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1])
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


In [ ]:
class StemEncoderCNN(nn.Module):
    """The stem/encoder/head architecture already used as PupilCNN in
    DualSensorFusion.ipynb and as PWFSNet in TwoStageAO.ipynb/
    05_TrainingAReconstructor.ipynb: each pupil image is processed
    independently in a grouped-conv stem before a shared encoder mixes
    information across pupils. Deeper than the original: the stem
    contributes one pooling stage, and num_downsamples adds as many more
    VGG-style stages as the remaining resolution allows."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=7):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(n_channels, base_channels * n_channels, kernel_size=11, padding=5, groups=n_channels),
            nn.GELU(),
            nn.Conv2d(base_channels * n_channels, 2 * base_channels * n_channels, kernel_size=7, padding=3, groups=n_channels),
            nn.GELU(),
            nn.MaxPool2d(2),
        )

        stem_out_channels = 2 * base_channels * n_channels
        n_stages = num_downsamples(Nout // 2)
        channels = [stem_out_channels] + [stem_out_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], activation=nn.GELU)
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)


In [ ]:
class CoordConvCNN(nn.Module):
    """ClassicCNN's exact conv-stage structure, with two extra channels --
    normalized x/y pixel-coordinate grids (CoordConv, Liu et al. 2018) --
    concatenated to the pupil channels before the first stage. Keeping every
    other layer identical to ClassicCNN isolates the ablation to "does
    positional information help", rather than confounding it with an
    unrelated conv-stack change."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=28):
        super().__init__()

        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, Nout), torch.linspace(-1, 1, Nout), indexing="ij"
        )
        self.register_buffer("coords", torch.stack([xx, yy]).unsqueeze(0))  # (1,2,Nout,Nout)

        n_stages = num_downsamples(Nout)
        channels = [n_channels + 2] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1])
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        coords = self.coords.expand(x.shape[0], -1, -1, -1)
        x = torch.cat([x, coords], dim=1)
        x = self.encoder(x)
        return self.head(x)


In [ ]:
class DetiledCNN(nn.Module):
    """Retiles the same 4 pupil crops FramePreprocess already produces into
    one 2x2 single-channel image (see retile_pupils above, and PyramidWFS's
    true quadrant layout), then runs a SimpleNet-style single-channel conv
    stack (AI4AO/PhaseEstimators.py) -- one stage deeper than the 4-channel
    architectures, since num_downsamples sees the retiled image's doubled
    (2*Nout) resolution."""

    def __init__(self, Nmodes, Nout, base_channels=14):
        super().__init__()

        retiled_size = 2 * Nout
        n_stages = num_downsamples(retiled_size)
        channels = [1] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1])
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = retile_pupils(x)
        x = self.encoder(x)
        return self.head(x)


In [ ]:
class RecommendedCNN(nn.Module):
    """StemEncoderCNN's grouped-conv stem (each pupil processed
    independently, keeping that processing physically meaningful) +
    CoordConvCNN's coordinate channels -- concatenated only where the stem's
    output first merges across pupils, not inside the grouped stem itself,
    since doing that would break the stem's groups=n_channels accounting --
    + a lightweight squeeze-and-excitation block (Hu et al. 2018) before the
    head, letting the network learn a data-dependent reweighting of the
    merged per-pupil features instead of relying purely on fixed conv
    weights to do that fusion."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=7, se_ratio=4):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(n_channels, base_channels * n_channels, kernel_size=11, padding=5, groups=n_channels),
            nn.GELU(),
            nn.Conv2d(base_channels * n_channels, 2 * base_channels * n_channels, kernel_size=7, padding=3, groups=n_channels),
            nn.GELU(),
            nn.MaxPool2d(2),
        )

        stem_out_size = Nout // 2
        stem_out_channels = 2 * base_channels * n_channels

        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, stem_out_size), torch.linspace(-1, 1, stem_out_size), indexing="ij"
        )
        self.register_buffer("coords", torch.stack([xx, yy]).unsqueeze(0))  # (1,2,stem_out_size,stem_out_size)

        n_stages = num_downsamples(stem_out_size)
        channels = [stem_out_channels + 2] + [stem_out_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], activation=nn.GELU)
        self.encoder = nn.Sequential(*layers)

        final_channels = channels[-1]
        se_hidden = max(final_channels // se_ratio, 4)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(final_channels, se_hidden),
            nn.ReLU(),
            nn.Linear(se_hidden, final_channels),
            nn.Sigmoid(),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(final_channels, Nmodes))

    def forward(self, x):
        x = self.stem(x)
        coords = self.coords.expand(x.shape[0], -1, -1, -1)
        x = torch.cat([x, coords], dim=1)
        x = self.encoder(x)

        gate = self.se(x).unsqueeze(-1).unsqueeze(-1)
        x = x * gate

        x = self.pool(x)
        return self.head(x)


In [ ]:
class ResBlock(nn.Module):
    """A basic ResNet residual block (no BatchNorm -- matching this repo's
    existing convention, since every reconstructor in AI4AO/PhaseEstimators.py
    skips it too): two 3x3 convs at a fixed resolution/channel count, with
    the block's own input added back before the final activation."""

    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.LeakyReLU()

    def forward(self, x):
        out = self.act(self.conv1(x))
        out = self.conv2(out)
        return self.act(out + x)


class ResNetCNN(nn.Module):
    """CoordConvCNN's input convention (plain, non-grouped convs + two
    coordinate channels) reorganized into ResNet-style residual stages: a
    stride-2 conv downsamples and changes channel count at the start of each
    stage, then `blocks_per_stage` ResBlocks refine the features at that
    resolution -- via a skip connection around each block's own two convs --
    before the next downsample."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=16, blocks_per_stage=2):
        super().__init__()

        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, Nout), torch.linspace(-1, 1, Nout), indexing="ij"
        )
        self.register_buffer("coords", torch.stack([xx, yy]).unsqueeze(0))

        n_stages = num_downsamples(Nout)
        stage_channels = [base_channels * 2 ** i for i in range(n_stages)]

        self.stem = nn.Sequential(
            nn.Conv2d(n_channels + 2, stage_channels[0], kernel_size=3, padding=1),
            nn.LeakyReLU(),
        )

        stages = []
        in_channels = stage_channels[0]
        for out_channels in stage_channels:
            stages.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1))
            stages.append(nn.LeakyReLU())
            for _ in range(blocks_per_stage):
                stages.append(ResBlock(out_channels))
            in_channels = out_channels
        self.stages = nn.Sequential(*stages)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(in_channels, Nmodes))

    def forward(self, x):
        coords = self.coords.expand(x.shape[0], -1, -1, -1)
        x = torch.cat([x, coords], dim=1)
        x = self.stem(x)
        x = self.stages(x)
        x = self.pool(x)
        return self.head(x)


## Instantiating all six

Each architecture is built once here and printed with its total trainable parameter count -- all six were sized (via each class's `base_channels` default) to land close to the same ~0.9M-parameter budget, so the closed-loop comparison at the end reflects architectural differences rather than one network simply having more capacity.

In [ ]:
architectures = {
    "Classic": ClassicCNN(n_channels, Nmodes, Nout).to(device=device),
    "StemEncoder": StemEncoderCNN(n_channels, Nmodes, Nout).to(device=device),
    "CoordConv": CoordConvCNN(n_channels, Nmodes, Nout).to(device=device),
    "Detiled": DetiledCNN(Nmodes, Nout).to(device=device),
    "Recommended": RecommendedCNN(n_channels, Nmodes, Nout).to(device=device),
    "ResNet": ResNetCNN(n_channels, Nmodes, Nout).to(device=device),
}

for name, model in architectures.items():
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:12s} -- total trainable parameters: {total_params:,}")


## Training

Following `05_TrainingAReconstructor.ipynb`'s exact procedure, repeated once per architecture: a fresh `AdamW` optimizer and a fresh `Trainer` per architecture, all sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)`.

**A deliberate trade-off, not an oversight:** `Trainer.train()` calls `self.dataset[0]` fresh at every outer step regardless of which `Trainer` instance is calling it (`PhaseDataset.__getitem__`'s `idx == 0` branch always redraws a new, statistically independent atmosphere). So training the six architectures one after another on the same shared `dataset` object does **not** give them the identical per-step atmosphere realization a hand-rolled shared-draw loop would -- each architecture instead trains on its own independent i.i.d. stream from the same distribution. This is fine for training itself, but it means the six *training-loss curves* below are directionally informative rather than a rigorously paired statistical test; only the seeded closed-loop rollout comparison further down is a true apples-to-apples comparison.

Six ~0.9M-parameter networks, each deeper than a shallow 2-pooling design, is a real wall-clock cost (more so than a first pass with fewer/shallower networks) -- consider a smaller `TrainParams['TrainRunNb']` for an initial correctness pass before committing to the full training budget.

In [ ]:
TrainParams['TrainRunNb'] = 5000
TrainParams['ClosedLoopIterations'] = 1

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=True)

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH + f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by
        # an earlier, differently-sized version of this architecture -- e.g.
        # after editing base_channels/depth above, old checkpoints under
        # PATH are no longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH + f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


## Comparing training-loss curves

`Trainer.plot_losses` only plots one tracker pair at a time, so we reimplement just its smoothing-and-overlay logic here for all six trackers at once. The oracle "ideal loss" bound (`z_inv` applied to the true residual OPD) is a property of the atmosphere/noise distribution and the DM, not of the architecture, so the six `loss_tracker_ideal` curves are statistically equivalent -- only one is shown, as a reference lower bound shared by all six.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for name, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=name)

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="--")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Loss")
ax.legend()
plt.show()


## Comparing closed-loop residual wavefront error (nm RMS)

For each architecture, reseed immediately before the rollout (`DualSensorFusion.ipynb`'s "reset the seed right before each rollout" trick) so all six see identical wind/r0/noise draws, then run `trainer.evaluate()` and compute the steady-state residual wavefront error in nm RMS over the pupil, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()` uses internally).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")


In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")

In [ ]:
names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]
param_counts = [sum(p.numel() for p in architectures[n].parameters() if p.requires_grad) for n in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(names, rms_values)
axes[0].set_ylabel("Steady-state residual (nm RMS)")
axes[0].set_title("Closed-loop performance")
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, param_counts)
axes[1].set_ylabel("Trainable parameters")
axes[1].set_title("Model size")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## Visualizing the best architecture's closed loop

A closer look at whichever architecture came out on top above -- the same rollout-animation pattern `05_TrainingAReconstructor.ipynb` uses.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best architecture: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes, 
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())
